# Contextual Compression for RAG
## A Hands-On Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/56-contextual-compression/contextual_compression_workbook.ipynb)

Standard RAG retrieves the top-k chunks and passes all of them to the LLM — including chunks that are topically related but don't actually answer the question. **Contextual compression** adds a filtering step: after retrieval, an LLM (or cross-encoder) reads each chunk and decides whether it contains information relevant to the specific query. Irrelevant chunks are dropped before generation. By the end of this workshop you will understand *why* noisy context hurts answer quality, *how* `LLMChainFilter` and `ContextualCompressionRetriever` work, and *how* to wire them into a LangGraph pipeline.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why noisy context hurts, what compression does |
| 2 | **Baseline** — Standard retrieval: retrieve top-8, pass all to LLM |
| 3 | **LLMChainFilter** — How the LLM-as-filter works |
| 4 | **Comparison** — Raw top-8 vs compressed docs |
| 5 | **Full Pipeline** — retrieve → compress → generate with LangGraph |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with `.venv` and `requirements.txt` installed, **or** Google Colab (install cell below)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> LangChain Contextual Compression: https://python.langchain.com/docs/how_to/contextual_compression/  
> Ma, X. et al. (2023). *Query Rewriting for Retrieval-Augmented Large Language Models.* https://arxiv.org/abs/2305.14283  
> Liu, N. et al. (2023). *Lost in the Middle: How Language Models Use Long Contexts.* https://arxiv.org/abs/2307.03172

In [ ]:
# Install dependencies — runs only on Google Colab.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "langchain",
        "langchain-openai",
        "langchain-chroma",
        "langchain-community",
        "chromadb",
        "langgraph",
        "python-dotenv",
    ])
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM or embedding cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — Why Noisy Context Hurts

---

### The Lost-in-the-Middle Problem

When you pass 8 retrieved chunks to an LLM, some of those chunks are relevant — and some are not. Research shows that LLMs:

1. **Attend strongly to the beginning and end** of the context window
2. **Miss information buried in the middle** (Liu et al. 2023 — "Lost in the Middle")
3. **Get confused by partially relevant chunks** that share vocabulary with the query but don't actually answer it

```
Context passed to LLM (8 chunks):

  [0] The transformer model introduced multi-head self-attention...     ← NOT relevant
  [1] RAG pipelines retrieve documents and pass them to the generator... ← NOT relevant  
  [2] Chunking splits long documents into smaller pieces...              ← NOT relevant
  [3] Contextual compression filters retrieved chunks...                 ← RELEVANT
  [4] LLMChainFilter asks the LLM to decide if a passage answers...      ← RELEVANT      
  [5] Cross-encoders can be used as compressors to score relevance...    ← RELEVANT
  [6] Embedding models map text to dense vectors...                      ← NOT relevant
  [7] The context window of GPT-4o-mini is 128k tokens...               ← NOT relevant

  → LLM must find the 3 relevant chunks among 8 total
  → Token cost: 8 chunks ≈ 640 tokens
```

After contextual compression:

```
  [3] Contextual compression filters retrieved chunks...                 ← KEPT
  [4] LLMChainFilter asks the LLM to decide if a passage answers...      ← KEPT
  [5] Cross-encoders can be used as compressors to score relevance...    ← KEPT

  → LLM receives only the 3 relevant chunks
  → Token cost: 3 chunks ≈ 240 tokens  (62% reduction)
```

---

### What Contextual Compression Does

```
QUERY: "What is contextual compression in RAG?"

RETRIEVE (standard):
  chunk_0, chunk_1, ..., chunk_7    ← top-8 by cosine similarity
       ↓
COMPRESS (new step):
  for each chunk:
      LLM or cross-encoder: "Is this passage relevant to the query?" → YES/NO
  keep only YES chunks
       ↓
GENERATE:
  LLM answers from compressed context only
```

The compressor can be:
- **`LLMChainFilter`** — asks the LLM itself to filter (flexible, but costs tokens per chunk)
- **`EmbeddingsFilter`** — computes embedding similarity of each chunk to the query (fast, no LLM call)
- **`LLMChainExtractor`** — asks the LLM to *rewrite* each chunk to only include the relevant sentence(s)
- **Cross-encoder** — scores each chunk with a local model (see Example 54)

In [ ]:
# Corpus and queries used throughout the workshop

SAMPLE_DOCS = [
    "The transformer model introduced multi-head self-attention for sequence modeling.",
    "RAG pipelines retrieve documents and pass them to the generator as context.",
    "Chunking splits long documents into smaller pieces that fit in a context window.",
    "Contextual compression filters retrieved chunks to keep only the relevant sentences.",
    "LLMChainFilter asks the LLM to decide if a passage answers the query.",
    "Cross-encoders can be used as compressors to score passage-level relevance.",
    "Embedding models map text to dense vectors for fast similarity search.",
    "The context window of GPT-4o-mini is 128k tokens, supporting long documents.",
    "Faithfulness measures whether an answer is supported by the provided context.",
    "Vector stores like ChromaDB index embeddings for approximate nearest-neighbor search.",
]

QUERIES = [
    "What is contextual compression in RAG pipelines?",
    "How does LLMChainFilter decide which passages to keep?",
    "What is the context window size of GPT-4o-mini?",
]

TOP_K = 8  # retrieve this many before compression

print(f"Corpus: {len(SAMPLE_DOCS)} documents | TOP_K={TOP_K}")
for i, d in enumerate(SAMPLE_DOCS):
    print(f"  [{i:02d}] {d}")

---

## Part 2 — Baseline: Standard Retrieval

---

First, build the vector store and run standard retrieval — no compression. This is our baseline. We retrieve `TOP_K=8` documents and pass all of them to the LLM.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_texts(
    SAMPLE_DOCS, embeddings, collection_name="compression-demo"
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

print(f"Index built: {len(SAMPLE_DOCS)} docs | retriever k={TOP_K}")

In [ ]:
from langchain_core.messages import HumanMessage

# ─── Baseline: retrieve top-8, generate without compression ──────────────────

def baseline_answer(query: str) -> tuple[list[str], str]:
    docs = base_retriever.invoke(query)
    texts = [d.page_content for d in docs]
    context = "\n".join(texts)
    msg = HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}")
    answer = llm.invoke([msg]).content
    return texts, answer


q = QUERIES[0]
raw_docs, baseline_ans = baseline_answer(q)

print(f"Query: '{q}'")
print(f"Retrieved {len(raw_docs)} chunks (no compression):")
for i, d in enumerate(raw_docs):
    print(f"  [{i}] {d}")
print()
print(f"Baseline answer: {baseline_ans}")

---

## Part 3 — LLMChainFilter

---

### How It Works

`LLMChainFilter` wraps the LLM into a binary filter. For each retrieved document, it constructs a prompt:

```
Given the following question and context, return 'YES' if the context is
relevant to answering the question and 'NO' if it is not.

Question: What is contextual compression in RAG pipelines?

Context: The transformer model introduced multi-head self-attention...

Answer:
```

Only documents that receive `YES` are passed downstream.

### `ContextualCompressionRetriever`

LangChain wraps both the base retriever and the compressor in a single object:

```python
compression_retriever = ContextualCompressionRetriever(
    base_compressor=LLMChainFilter.from_llm(llm),
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": TOP_K}),
)

# One call does both: retrieve top-8 then filter
compressed_docs = compression_retriever.invoke(query)
```

**Cost note:** `LLMChainFilter` makes one LLM call per retrieved document. For `TOP_K=8`, that's 8 short LLM calls per query. Use `EmbeddingsFilter` instead when cost matters more than precision.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainFilter

compressor = LLMChainFilter.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)
print("ContextualCompressionRetriever ready.")
print(f"  Base retriever: top-{TOP_K} by cosine similarity")
print(f"  Compressor: LLMChainFilter (gpt-4o-mini binary YES/NO per chunk)")

In [ ]:
# ─── Run compression on all three queries ────────────────────────────────────
# Note: this makes 8 LLM calls per query (one per chunk)

results: dict[str, dict] = {}

for query in QUERIES:
    raw = base_retriever.invoke(query)
    compressed = compression_retriever.invoke(query)
    results[query] = {
        "raw": [d.page_content for d in raw],
        "compressed": [d.page_content for d in compressed],
    }

# Summary
print(f"{'Query':<50} {'Raw':>5} {'Compressed':>10} {'Filtered':>8}")
print("-" * 78)
for q, r in results.items():
    n_raw = len(r["raw"])
    n_comp = len(r["compressed"])
    print(f"{q[:48]:<50} {n_raw:>5} {n_comp:>10} {n_raw - n_comp:>8}")

---

## Part 4 — Comparison: Raw vs Compressed

---

For each query, we compare:
- Which documents were retrieved by the base retriever
- Which survived compression
- Whether the surviving documents are genuinely relevant

In [ ]:
# ─── Side-by-side: which docs are kept vs filtered ───────────────────────────

for query, r in results.items():
    kept = set(r["compressed"])
    print(f"Query: '{query}'")
    print(f"  {'Status':<8}  Document")
    print(f"  {'------':<8}  {'─'*60}")
    for doc in r["raw"]:
        status = "KEPT   " if doc in kept else "filtered"
        print(f"  {status:<8}  {doc}")
    print()

In [ ]:
# ─── Compare answers: baseline vs compressed ─────────────────────────────────

for query, r in results.items():
    # Baseline answer (all 8 chunks)
    ctx_raw = "\n".join(r["raw"])
    ans_raw = llm.invoke([HumanMessage(
        content=f"Context:\n{ctx_raw}\n\nQuestion: {query}"
    )]).content

    # Compressed answer
    ctx_comp = "\n".join(r["compressed"]) if r["compressed"] else "No relevant context."
    ans_comp = llm.invoke([HumanMessage(
        content=f"Context:\n{ctx_comp}\n\nQuestion: {query}"
    )]).content

    print(f"QUERY: {query}")
    print(f"  Chunks in baseline : {len(r['raw'])} chunks, {len(ctx_raw)} chars")
    print(f"  Chunks compressed  : {len(r['compressed'])} chunks, {len(ctx_comp)} chars")
    print(f"  Baseline answer    : {ans_raw}")
    print(f"  Compressed answer  : {ans_comp}")
    print()

---

## Part 5 — Full Pipeline with LangGraph

---

Three nodes:

```
START
  │
  ▼
retrieve   ─── Chroma: top-8 by cosine similarity → raw_docs
  │
  ▼
compress   ─── LLMChainFilter: YES/NO per doc → compressed
  │
  ▼
generate   ─── GPT-4o-mini answers from compressed context
  │
  ▼
 END
```

Note: `retrieve` and `compress` are separate nodes rather than using `ContextualCompressionRetriever` directly. This lets us inspect the intermediate state (`raw_docs`) in the graph result.

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class CompressionState(TypedDict):
    query:      str
    raw_docs:   list
    compressed: list
    answer:     str


def retrieve_node(state: CompressionState) -> dict:
    docs = base_retriever.invoke(state["query"])
    return {"raw_docs": [d.page_content for d in docs]}


def compress_node(state: CompressionState) -> dict:
    docs = compression_retriever.invoke(state["query"])
    return {"compressed": [d.page_content for d in docs]}


def generate_node(state: CompressionState) -> dict:
    context = (
        "\n".join(state["compressed"])
        if state["compressed"]
        else "No relevant context found."
    )
    msg = HumanMessage(
        content=f"Answer using ONLY the context below.\n\nContext:\n{context}\n\nQuestion: {state['query']}"
    )
    return {"answer": llm.invoke([msg]).content}


graph = StateGraph(CompressionState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("compress", compress_node)
graph.add_node("generate", generate_node)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "compress")
graph.add_edge("compress", "generate")
graph.add_edge("generate", END)
app = graph.compile()

print("Pipeline compiled: retrieve → compress → generate")

In [ ]:
# ─── Run all three queries through the full pipeline ─────────────────────────

for query in QUERIES:
    print(f"{'=' * 65}")
    print(f"Query: {query}")
    result: CompressionState = app.invoke(
        {"query": query, "raw_docs": [], "compressed": [], "answer": ""}
    )
    saved = len(result["raw_docs"]) - len(result["compressed"])
    print(f"Raw docs  : {len(result['raw_docs'])} retrieved")
    print(f"Compressed: {len(result['compressed'])} kept ({saved} filtered out)")
    print(f"Answer    : {result['answer']}")
    print()

---

### Exercise 1 — Try EmbeddingsFilter

**Goal:** Replace `LLMChainFilter` with `EmbeddingsFilter` from `langchain.retrievers.document_compressors`. `EmbeddingsFilter` keeps only documents whose embedding cosine similarity to the query exceeds a threshold (no LLM calls — much faster and cheaper).

```python
from langchain.retrievers.document_compressors import EmbeddingsFilter
emb_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.75)
```

Run all three queries with `similarity_threshold=0.75` and compare the results to `LLMChainFilter`. Which compressor is more conservative (keeps fewer docs)?

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
from langchain.retrievers.document_compressors import EmbeddingsFilter

# TODO: create emb_filter with similarity_threshold=0.75
# TODO: create a ContextualCompressionRetriever using emb_filter
# TODO: run all 3 queries and print: kept count for EmbeddingsFilter vs LLMChainFilter
pass

### Exercise 2 — What Happens When Nothing Is Relevant?

**Goal:** Run the pipeline with this query that is entirely off-topic for the corpus:

```python
OFF_TOPIC = "What is the boiling point of water at high altitude?"
```

1. How many documents does the base retriever return?
2. How many survive compression?
3. What does the `generate` node answer when `compressed` is empty?

**Hint:** Look at how `generate_node` handles the empty case in the pipeline code above.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────

OFF_TOPIC = "What is the boiling point of water at high altitude?"

# TODO: invoke app with OFF_TOPIC, print raw_docs count, compressed count, and answer
pass

---

## What's Next?

You now understand contextual compression: retrieve broad, then filter precisely.

### Stack compression with other techniques
- **Example 54 — Reranking RAG** ([`../54-reranking-rag/reranking_workbook.ipynb`](../54-reranking-rag/reranking_workbook.ipynb)): use a cross-encoder as the compressor instead of `LLMChainFilter` — same idea, no LLM call needed
- **Example 55 — HyDE** ([`../55-hyde-rag/hyde_workbook.ipynb`](../55-hyde-rag/hyde_workbook.ipynb)): combine HyDE (better recall at Stage 1) with compression (better precision at Stage 2)
- **Example 57 — Step-Back Prompting** ([`../57-step-back-prompting/`](../57-step-back-prompting/README.md)): another query-rewriting technique for improving retrieval on specific factual questions

### Compressor options in LangChain
- `LLMChainExtractor` — rewrites each chunk to contain only the relevant sentence(s), not just YES/NO
- `EmbeddingsFilter` — no LLM call; fast and cheap for high-volume applications
- `DocumentCompressorPipeline` — chain multiple compressors in sequence

### Evaluate your pipeline
- **Example 16 — RAG Evaluation** ([`../16-rag-eval-notebook/`](../16-rag-eval-notebook/README.md)): measure Context Recall before and after compression — does compression hurt recall?

---

*Workshop complete. Next step: Example 57 — step-back prompting to improve retrieval recall.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself.

### Exercise 1 — EmbeddingsFilter vs LLMChainFilter

**Expected behaviour:** `EmbeddingsFilter` at threshold `0.75` typically keeps *fewer* documents than `LLMChainFilter` because:
- Embeddings measure semantic similarity at the *surface* level
- LLMChainFilter uses the full LLM reasoning capacity to understand whether a passage is *functionally* relevant to answering the question

For query `"How does LLMChainFilter decide which passages to keep?"`, the `EmbeddingsFilter` might filter out the cross-encoder document because it doesn't contain the word "LLMChainFilter", even though it's conceptually related. `LLMChainFilter` would likely keep it.

In [ ]:
# Exercise 1 — sample solution
from langchain.retrievers.document_compressors import EmbeddingsFilter

emb_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.75)
emb_compression_retriever = ContextualCompressionRetriever(
    base_compressor=emb_filter,
    base_retriever=base_retriever,
)

print(f"{'Query':<50} {'LLMChain':>8} {'EmbFilter':>9}")
print("-" * 70)

for query in QUERIES:
    llm_docs = compression_retriever.invoke(query)
    emb_docs = emb_compression_retriever.invoke(query)
    print(f"{query[:48]:<50} {len(llm_docs):>8} {len(emb_docs):>9}")

### Exercise 2 — Off-Topic Query

**Expected behaviour:**
- Base retriever still returns `TOP_K=8` documents (it always returns k results)
- LLMChainFilter scores all 8 as `NO` and returns an empty list
- The `generate_node` receives `compressed=[]` and falls back to the message `"No relevant context found."`
- The LLM answers `"No relevant context found."` or a variation of `"I cannot answer this question based on the provided context."`

This graceful degradation is intentional — it prevents the LLM from hallucinating an answer when the corpus doesn't contain relevant information.

In [ ]:
# Exercise 2 — sample solution

OFF_TOPIC = "What is the boiling point of water at high altitude?"

result = app.invoke(
    {"query": OFF_TOPIC, "raw_docs": [], "compressed": [], "answer": ""}
)

print(f"Query: '{OFF_TOPIC}'")
print(f"Raw retrieved  : {len(result['raw_docs'])} docs")
print(f"After compress : {len(result['compressed'])} docs")
print(f"Answer         : {result['answer']}")